# Feature Selection

### Motivation: Feature selection was performed jointly across the training and validation cohorts to ensure that only features measurable in both datasets were retained for modelling. Since the training data (CheckMate, RNA-seq) and validation data (E-MTAB-3218, microarray) were generated on different platforms, not all gene signature scores were available in both cohorts. Restricting the feature set to the common subset between the two datasets prevents the model from relying on features that cannot be evaluated at validation, ensuring a fair and unbiased assessment of generalisation performance.

In [1]:
# ── Imports ────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from typing import Literal
from typing import Literal
from scipy import stats
import re
from collections import Counter


# ── Project utilities ──────────────────────────────────────────────────────
from src.utils.config import load_config
from src.utils.helper_func import find_common_features


In [2]:
args = {"config":"configs/Params_set.yml"}

cfg = load_config(args["config"])


In [3]:
pd.set_option('display.max_columns', None)

### Part A. Cohort Loading

In [4]:
# Loading training and validation cohort
chkmate_cohort = pd.read_excel(cfg["Features"]["Checkmate"], sheet_name="Features")
emtab_cohort = pd.read_excel(cfg["Features"]["Emtab"], sheet_name="Features")

In [5]:
chkmate_cohort.head(5)

,SUBJID,Cohort,Arm,MAF_Tumor_ID,MAF_Normal_ID,CNV_ID,RNA_ID,CD8_IF_ID,Sex,Age,MSKCC,IMDC,Sarc,Rhab,Sarc_or_Rhab,Received_Prior_Therapy,Number_of_Prior_Therapies,Days_from_TumorSample_Collection_and_Start_of_Trial_Therapy,SampleType,Tumor_Sample_Primary_or_Metastasis,Site_of_Metastasis,Tumor_Shrinkage,ORR,Benefit,ExtremeResponder,PFS,PFS_CNSR,OS,OS_CNSR,irORR,irPFS,irPFS_CNSR,Purity,Ploidy,TM_Area,TM_CD8,TM_CD8_Density,TC_Area,TC_CD8,TC_CD8_Density,TM_TC_Ratio,ImmunoPhenotype,TM_CD8_PERCENT,TC_CD8_PERCENT,TMB_Counts,FS_Counts,NEO_Weak,WGII,ITH,homozygous_ANY,Amplification_1p36.32,Amplification_1q21.3,Amplification_1q32.1,Amplification_2q37.3,Amplification_5p15.33,Amplification_5q31.3,Amplification_5q35.3,Amplification_6p21.2,Amplification_6q21,Amplification_7p22.2,Amplification_7q36.2,Amplification_8p11.21,Amplification_8q24.3,Amplification_9q34.11,Amplification_11q13.3,Amplification_11q14.1,Amplification_12q24.32,Amplification_17q21.2,Amplification_17q25.3,Amplification_20q13.33,Amplification_21q22.3,Deletion_1p36.31,Deletion_1p36.11,Deletion_1q42.3,Deletion_2q37.1,Deletion_3p21.1,Deletion_5q14.1,Deletion_6p22.2,Deletion_6p21.32,Deletion_6p12.1,Deletion_6q25.2,Deletion_8p23.2,Deletion_9p21.3,Deletion_9q34.3,Deletion_10q23.31,Deletion_10q26.3,Deletion_11q12.3,Deletion_11q23.1,Deletion_12q24.33,Deletion_14q32.33,Deletion_15q11.2,Deletion_19p13.3,Deletion_19q13.42,Deletion_22q11.21,ARID1A,ATM,BAP1,COL9A3,KDM5C,MTOR,NF2,PBRM1,PCK1,PIK3CA,PTEN,S100B,SETD2,SMARCA4,TCEB1,TP53,TRMT2B,TSC1,USP32,VHL,WNT8A,ZNF800,Angio,Teff,Myeloid,Javelin,Merck18,Response,T_effector,Angiogenesis,FAO/AMPK,Cell_Cycle,FAS/Pentose_Phosphate,Stroma,Myeloid_Inflammation,Complement_Cascade,Omega_oxidation,SnoRNA,Tumor_inflammation_score
0,RCC25-582,CM-025,NIVOLUMAB,EA699646,EA699596,EA699646.called.seg,EA595500,NaN,Male,61,INTERMEDIATE,INTERMEDIATE,0.0,0.0,0.0,Yes,1.0,467.0,FFPE,PRIMARY,NaN,14.473684,PD,NCB,PD,2.135524,1,3.548255,1,NaN,NaN,NaN,0.49,1.88,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No_IF,NaN,NaN,105.0,12.0,370.0,0.183607,0.398551,1.0,WT,WT,WT,WT,MUT,MUT,MUT,WT,WT,WT,WT,WT,WT,WT,MUT,MUT,WT,WT,WT,WT,MUT,MUT,WT,WT,MUT,MUT,WT,WT,WT,WT,WT,MUT,MUT,MUT,WT,MUT,WT,WT,WT,MUT,MUT,MUT,MUT,MUT,WT,WT,WT,WT,WT,WT,WT,MUT,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT,34.494799,30.257235,27.933528,30.244353,32.096130,Non-Responders,0.148237,0.207268,0.426179,-0.718344,-0.802468,-0.972039,-1.191040,-0.094806,0.388907,-0.313596,-0.236995
1,RCC25-624,CM-025,NIVOLUMAB,EA699986,EA699947,NaN,EA595597,EA021689,Female,64,INTERMEDIATE,INTERMEDIATE,0.0,0.0,0.0,Yes,1.0,NaN,FFPE,METASTASIS,OTHER,-100.000000,CRPR,CB,ER,53.716632,0,57.396304,0,NaN,NaN,NaN,NaN,NaN,20.02048,6819.0,340.601224,103.05978,58613.0,568.728169,0.598882,Infiltrated,12.67260,17.62440,28.0,0.0,135.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT,33.908834,32.175234,32.144320,32.374054,34.035515,Responders,1.174420,-0.074041,-1.016133,0.081855,-0.003479,1.047075,0.809109,0.484398,-0.572133,-0.313596,1.030557
2,RCC25-982,CM-025,NIVOLUMAB,EA698460,EA698424,EA698460.called.seg,EA595624,L1381831,Male,67,INTERMEDIATE,INTERMEDIATE,0.0,0.0,0.0,Yes,1.0,NaN,FFPE,PRIMARY,NaN,-100.000000,CRPR,CB,NEITHER,11.827515,1,17.839836,0,NaN,NaN,NaN,0.33,3.49,21.74988,1588.0,73.011897,69.63336,3116.0,44.748666,1.631599,Desert,1.40794,0.77014,181.0,6.0,882.0,0.035993,0.098485,0.0,WT,WT,WT,MUT,MUT,MUT,MUT,WT,WT,MUT,MUT,WT,WT,WT,WT,WT,MUT,WT,WT,WT,MUT,WT,WT,WT,WT,MUT,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT,MUT,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT,MUT,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT,35.694310,29.096607,28.023430,29.692392,31.275094,Responders,-0.260588,0.973303,1.286888,-0.856730,-0.832729,-0.669412,-1.164171,-1.296157,-0.321287,-0.313596,-0.652298
3,RCC25-493,CM-025,NIVOLUMAB,NaN,NaN,NaN,EA595635,L1293281,Male,65,INTERMEDIATE,INTERMEDIATE,0.

In [6]:
chk_patient_id = chkmate_cohort["SUBJID"].str.extract(r"[-_](\d+)$")

In [7]:
emtab_cohort.head(5)

,Source Name,organism,disease,individual,sex,clinical history,biopsy timepoint,clinical trial arm,BOR,BOR3,MPCT,MPCTrank,Response20pct,OS,PFS,OS_SOR,PFS_SOR,TMB,CD8PERCENT BL,CD4PERCENT BL,PD1PERCENT BL,PDL1 PERCENT,PDL1cat,TP_strelka,TP_Tnsnv,TH_strelka_cluster_count,TH_Tnsnv_cluster_count,Subject_Affy_Status,MatchedLesionBiopsy,PBRM1,VHL,BAP1,SETD2,BiopsySite,Biopsy Lesion,MPCTBiopsy,Material Type,Protocol REF,Extract Name,Protocol REF.1,Labeled Extract Name,Label,Protocol REF.2,Assay Name,Technology Type,Array Design REF,Term Source REF,Protocol REF.3,Array Data File,Comment [ArrayExpress FTP file],Factor Value[compound],Factor Value[dose],Unit[concentration unit],PFS_months,OS_months,Response,T_effector,Angiogenesis,FAO/AMPK,Cell_Cycle,FAS/Pentose_Phosphate,Stroma,Myeloid_Inflammation,Complement_Cascade,Omega_oxidation,SnoRNA,Tumor_inflammation_score
0,14_PRE,Homo sapiens,clear cell renal carcinoma,14,male,PriorTherapy_No,Screen,D,PR,CRPR,-86.486486,2,20pctDec,1025,541,1,0,56.0,NaN,NaN,NaN,0.0,0%,48.915050,39.998361,4.0,2.0,17Pre,False,MUT,WT,WT,WT,LymphNode,C,NaN,organism part,P-MTAB-42749,14_PRE,P-MTAB-42745,14_PRE:biotin,biotin,P-MTAB-42746,5500994154005050413582_A01,array assay,A-GEOD-19099,ArrayExpress,P-MTAB-42748,5500994154005050413582_A01.cel,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,nivolumab,10.0,milligram per kilogram,17.774127,33.675565,Responders,-1.249629,-1.213285,0.601988,0.093063,1.079233,-0.277245,0.162433,0.091752,0.017431,0,-1.271906
1,105_PRE,Homo sapiens,clear cell renal carcinoma,105,male,PriorTherapy_Yes,Screen,C,PD,PD,16.666667,68,Not20pct,118,45,0,0,72.0,0.13,0.00,0.42,0.0,0%,70.122411,70.050709,4.0,3.0,42Pairs,False,WT,WT,WT,WT,SoftTissue,K,NaN,organism part,P-MTAB-42749,105_PRE,P-MTAB-42745,105_PRE:biotin,biotin,P-MTAB-42746,5500994154005050413582_A02,array assay,A-GEOD-19099,ArrayExpress,P-MTAB-42748,5500994154005050413582_A02.cel,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,nivolumab,10.0,milligram per kilogram,1.478439,3.876797,Non-Responders,-0.575128,1.039385,0.443900,-0.820964,-0.471185,0.603002,-0.547749,-0.615865,-0.642438,0,-0.641897
2,22_PRE,Homo sapiens,clear cell renal carcinoma,22,male,PriorTherapy_Yes,Screen,B,PR,CRPR,-48.958333,13,20pctDec,197,197,0,0,21.0,35.26,0.00,2.71,30.0,5+%,31.145284,11.109099,3.0,1.0,17Pre,True,WT,WT,WT,WT,SoftTissue,M,NaN,organism part,P-MTAB-42749,22_PRE,P-MTAB-42745,22_PRE:biotin,biotin,P-MTAB-42746,5500994154005050413582_A03,array assay,A-GEOD-19099,ArrayExpress,P-MTAB-42748,5500994154005050413582_A03.cel,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,nivolumab,2.0,milligram per kilogram,6.472279,6.472279,Responders,0.294987,-0.325934,-0.015568,-1.462302,0.096124,0.470951,-0.696113,0.040591,-0.028226,0,0.047626
3,1_PRE,Homo sapiens,clear cell renal carcinoma,1,male,PriorTherapy_Yes,Screen,C,PD,PD,24.342105,74,Not20pct,43,38,1,0,34.0,1.16,0.00,0.51,0.0,0%,44.534099,32.933363,4.0,2.0,42Pairs,False,WT,MUT,WT,WT,LymphNode,U,NaN,organism part,P-MTAB-42749,1_PRE,P-MTAB-42745,1_PRE:biotin,biotin,P-MTAB-42746,5500994154005050413582_A08,array assay,A-GEOD-19099,ArrayExpress,P-MTAB-42748,5500994154005050413582_A08.cel,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,nivolumab,10.0,milligram per kilogram,1.248460,1.412731,Non-Responders,0.196749,-0.213276,1.201464,-0.605462,0.060638,-1.477577,-0.007022,0.858854,0.797571,0,0.538887
4,66_PRE,Homo sapiens,clear cell renal carcinoma,66,male,PriorTherapy_Yes,Screen,B,PD,PD,37.837838,78,Not20pct,470,36,0,0,NaN,20.28,0.34,30.33,0.0,0%,19.357211,51.729648,1.0,1.0,17Pre,True,NaN,NaN,NaN,NaN,Adrenal,D,18.918919,organism part,P-MTAB-42749,66_PRE,P-MTAB-42745,66_PRE:biotin,biotin,P-MTAB-42746,5500994154005050413582_B02,array assay,A-GEOD-19099,ArrayExpress,P-MTAB-42748,5500994154005050413582_B02.cel,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,nivolumab,2.0,milligram per kilogram,1.182752,15.441478,Non-Responders,-0.637247,-0.074359,-0.644113,-0.106279,-0.460072,0.501169,0.737388,0.209353,0.074781,0,0.0

In [8]:
emtab_cohort = emtab_cohort.rename(columns={'clinical history':'Received_Prior_Therapy', "sex":"Sex", "TMB":"TMB_Counts"})

In [9]:
emtab_cohort["Received_Prior_Therapy"] = emtab_cohort["Received_Prior_Therapy"].str.replace("PriorTherapy_", "")

In [10]:
res = find_common_features(chkmate_cohort.columns,emtab_cohort.columns )

### Part B. Selecting Common Features

In [11]:
Common_features = res["exact"]
Common_features = [i[0] for i in Common_features]
rm = ["Sex", "Benefit","PFS", "OS", "SnoRNA"]
Common_features = [x for x in Common_features if x not in rm]

In [12]:
Common_features

['Received_Prior_Therapy',
 'TMB_Counts',
 'BAP1',
 'PBRM1',
 'SETD2',
 'VHL',
 'Response',
 'T_effector',
 'Angiogenesis',
 'FAO/AMPK',
 'Cell_Cycle',
 'FAS/Pentose_Phosphate',
 'Stroma',
 'Myeloid_Inflammation',
 'Complement_Cascade',
 'Omega_oxidation',
 'Tumor_inflammation_score']

In [13]:
Final_chkmate_features = chkmate_cohort[Common_features]
Final_emtab_features = emtab_cohort[Common_features]

In [14]:
Final_chkmate_features["Cohort"] = chkmate_cohort["Cohort"]

/var/folders/cg/zz8qcxpj5x9_g4wb8khq9t000000gn/T/ipykernel_74127/648240579.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Final_chkmate_features["Cohort"] = chkmate_cohort["Cohort"]


### Part C. Final Selected Features

In [15]:
Final_chkmate_features.head(5)

,Received_Prior_Therapy,TMB_Counts,BAP1,PBRM1,SETD2,VHL,Response,T_effector,Angiogenesis,FAO/AMPK,Cell_Cycle,FAS/Pentose_Phosphate,Stroma,Myeloid_Inflammation,Complement_Cascade,Omega_oxidation,Tumor_inflammation_score,Cohort
0,Yes,105.0,WT,MUT,WT,WT,Non-Responders,0.148237,0.207268,0.426179,-0.718344,-0.802468,-0.972039,-1.191040,-0.094806,0.388907,-0.236995,CM-025
1,Yes,28.0,WT,WT,WT,WT,Responders,1.174420,-0.074041,-1.016133,0.081855,-0.003479,1.047075,0.809109,0.484398,-0.572133,1.030557,CM-025
2,Yes,181.0,WT,MUT,WT,WT,Responders,-0.260588,0.973303,1.286888,-0.856730,-0.832729,-0.669412,-1.164171,-1.296157,-0.321287,-0.652298,CM-025
3,Yes,NaN,NaN,NaN,NaN,NaN,Non-Responders,0.035187,-0.310847,0.106104,0.490507,1.029595,1.261651,0.551131,0.209985,-0.929884,0.443611,CM-025
4,Yes,NaN,NaN,NaN,NaN,NaN,Responders,1.560963,-0.515835,-0.646998,0.809698,-0.208788,0.236488,0.165933,1.066020,0.568143,0.553803,CM-025


In [16]:
Final_emtab_features.head(5)

,Received_Prior_Therapy,TMB_Counts,BAP1,PBRM1,SETD2,VHL,Response,T_effector,Angiogenesis,FAO/AMPK,Cell_Cycle,FAS/Pentose_Phosphate,Stroma,Myeloid_Inflammation,Complement_Cascade,Omega_oxidation,Tumor_inflammation_score
0,No,56.0,WT,MUT,WT,WT,Responders,-1.249629,-1.213285,0.601988,0.093063,1.079233,-0.277245,0.162433,0.091752,0.017431,-1.271906
1,Yes,72.0,WT,WT,WT,WT,Non-Responders,-0.575128,1.039385,0.443900,-0.820964,-0.471185,0.603002,-0.547749,-0.615865,-0.642438,-0.641897
2,Yes,21.0,WT,WT,WT,WT,Responders,0.294987,-0.325934,-0.015568,-1.462302,0.096124,0.470951,-0.696113,0.040591,-0.028226,0.047626
3,Yes,34.0,WT,WT,WT,MUT,Non-Responders,0.196749,-0.213276,1.201464,-0.605462,0.060638,-1.477577,-0.007022,0.858854,0.797571,0.538887
4,Yes,NaN,NaN,NaN,NaN,NaN,Non-Responders,-0.637247,-0.074359,-0.644113,-0.106279,-0.460072,0.501169,0.737388,0.209353,0.074781,0.007184


In [17]:
with pd.ExcelWriter("data/processed/final_chkmate_features.xlsx", engine="openpyxl") as writer:
    Final_chkmate_features.to_excel(writer, sheet_name="Features", index=False)

with pd.ExcelWriter("data/processed/final_emtab_features.xlsx", engine="openpyxl") as writer:
    Final_emtab_features.to_excel(writer, sheet_name="Features", index=False)